In [1]:
%pip install -q sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Imports
from pathlib import Path
from pypdf import PdfReader
from langchain_core.documents import Document

# Load the law page by page and the transcript as one Document
reader = PdfReader("eu_ai_act.pdf")
law_docs = [
    Document(page_content=page.extract_text() or "", metadata={"source": "law", "page": page_number})
    for page_number, page in enumerate(reader.pages, start=1)
]
transcript_doc = Document(
    page_content=Path("transcripcion_clase_27oct.txt").read_text(encoding="utf-8"),
    metadata={"source": "class_transcript"},
)
documents = law_docs + [transcript_doc]

# Print counts, metadata, and previews
for source_name in ("law", "class_transcript"):
    print(f"{source_name}: {sum(doc.metadata['source'] == source_name for doc in documents)}")
print("Law page 1:", law_docs[0].metadata, law_docs[0].page_content[:200])
print("Transcript:", transcript_doc.metadata, transcript_doc.page_content[:200])


law: 144
class_transcript: 1
Law page 1: {'source': 'law', 'page': 1} REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL
of 13 June 2024
laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, 
(EU) No 167/
Transcript: {'source': 'class_transcript'} 00:16:13 The relevant document, okay
00:16:15 Ami
00:16:17 I'm also thinking how we applied in a general purpose, for example, chatbot.
00:16:26 Uh, is it that because now in this case, we got the que


In [3]:
# Split Documents while keeping their metadata
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)

# Print counts, a law preview, and the maximum chunk length
for source_name in ("law", "class_transcript"):
    print(f"{source_name}: {sum(chunk.metadata['source'] == source_name for chunk in chunks)}")
law_chunk = next(chunk for chunk in chunks if chunk.metadata["source"] == "law")
print("Law chunk:", law_chunk.metadata, law_chunk.page_content[:200])
print("Maximum chunk length:", max(len(chunk.page_content) for chunk in chunks))


law: 775
class_transcript: 153
Law chunk: {'source': 'law', 'page': 1} REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL
of 13 June 2024
laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, 
(EU) No 167/
Maximum chunk length: 999


In [4]:
# Load the model and encode chunk text in the existing order
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embedding_model.encode(
    [chunk.page_content for chunk in chunks],
    normalize_embeddings=True,
)

# Print counts, vector size, and a preview
print("Chunks:", len(chunks))
print("Vectors:", len(chunk_embeddings))
print("Numbers per vector:", len(chunk_embeddings[0]))
print("First vector (first 5):", chunk_embeddings[0][:5])


c:\Users\jaime\anaconda3\envs\bootcamp-semana2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16702.62it/s]


Chunks: 928
Vectors: 928
Numbers per vector: 384
First vector (first 5): [-0.01454     0.0169065   0.00171308 -0.06310635  0.0291541 ]


In [5]:
# Search by cosine similarity using the normalized dot product
def search(query, k=5):
    query_embedding = embedding_model.encode(query, normalize_embeddings=True)
    similarities = chunk_embeddings @ query_embedding
    top_indices = similarities.argsort()[::-1][:k]
    return [(chunks[index], float(similarities[index])) for index in top_indices]

# Run the example query and print ranked results
results = search("Which AI practices are prohibited?")
for position, (chunk, score) in enumerate(results, start=1):
    source = chunk.metadata.get("source", "-")
    page = chunk.metadata.get("page", "-")
    print(f"{position}. score={score:.3f} | source={source} | page={page} | {chunk.page_content[:200]}")


1. score=0.714 | source=law | page=51 | Article 4
AI literacy
Providers and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of 
their staff and other persons dealing with the op
2. score=0.683 | source=law | page=6 | benefits, risks, safeguards, rights and obligations in relation to the use of AI systems. In cooperation with the relevant 
stakeholders, the Commission and the Member States should facilitate the dra
3. score=0.681 | source=law | page=7 | conducted in accordance with applicable Union law.
(26) In order to introduce a proportionate and effective set of binding rules for AI systems, a clearly defined risk-based 
approach should be follow
4. score=0.677 | source=law | page=14 | methodology and criteria envisaged also for any future amendments of the list of high-risk AI systems that the 
Commission should be empowered to adopt, via delegated acts, to take into account the ra
5. score=0.669 | source=law | page=51 

In [6]:
# Load the cross-encoder reranker
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Score search candidates and keep each candidate's original search rank
def rerank(query, k_candidates=20, k_final=5):
    candidates = search(query, k=k_candidates)
    pairs = [(query, chunk.page_content) for chunk, _ in candidates]
    rerank_scores = reranker.predict(pairs)
    ranked = [
        (chunk, float(rerank_score), similarity, search_position)
        for search_position, ((chunk, similarity), rerank_score)
        in enumerate(zip(candidates, rerank_scores), start=1)
    ]
    return sorted(ranked, key=lambda result: result[1], reverse=True)[:k_final]

# Run the example query and print reranked results
results = rerank("Which AI practices are prohibited?")
for new_position, (chunk, rerank_score, similarity, search_position) in enumerate(results, start=1):
    source = chunk.metadata.get("source", "-")
    page = chunk.metadata.get("page", "-")
    print(f"{new_position}. search={search_position} | reranker={rerank_score:.3f} | similarity={similarity:.3f} | source={source} | page={page} | {chunk.page_content[:200]}")


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4182.75it/s]


1. search=1 | reranker=8.004 | similarity=0.714 | source=law | page=51 | Article 4
AI literacy
Providers and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of 
their staff and other persons dealing with the op
2. search=9 | reranker=6.734 | similarity=0.663 | source=law | page=9 | intention to distort behaviour where the distortion results from factors external to the AI system which are outside 
the control of the provider or the deployer, namely factors that may not be reason
3. search=16 | reranker=4.998 | similarity=0.648 | source=law | page=12 | generalisability. Therefore, AI systems identifying or inferring emotions or intentions of natural persons on the basis 
of their biometric data may lead to discriminatory outcomes and can be intrusiv
4. search=3 | reranker=3.447 | similarity=0.681 | source=law | page=7 | conducted in accordance with applicable Union law.
(26) In order to introduce a proportionate and effectiv

In [7]:
# Add section metadata to each chunk using its source and page
for chunk in chunks:
    if chunk.metadata["source"] == "law":
        page = chunk.metadata["page"]
        chunk.metadata["section"] = "recital" if page <= 43 else "article" if page <= 123 else "annex"
    else:
        chunk.metadata["section"] = "class"

# Count chunks by section
for section in ("recital", "article", "annex", "class"):
    print(f"{section}: {sum(chunk.metadata['section'] == section for chunk in chunks)}")

# Search only matching chunks while keeping each vector paired with its chunk
def search_filtered(query, k=5, filters=None):
    eligible = [
        (chunk, vector) for chunk, vector in zip(chunks, chunk_embeddings)
        if all(chunk.metadata.get(key) == value for key, value in (filters or {}).items())
    ]
    if not eligible:
        return []
    query_vector = embedding_model.encode(query, normalize_embeddings=True)
    scored = [(chunk, float(vector @ query_vector)) for chunk, vector in eligible]
    return sorted(scored, key=lambda result: result[1], reverse=True)[:k]

# Rerank filtered search results and retain their search positions
def rerank_filtered(query, filters=None, k_candidates=20, k_final=5):
    candidates = search_filtered(query, k=k_candidates, filters=filters)
    pairs = [(query, chunk.page_content) for chunk, _ in candidates]
    rerank_scores = reranker.predict(pairs) if pairs else []
    ranked = [
        (chunk, float(rerank_score), similarity, search_position)
        for search_position, ((chunk, similarity), rerank_score)
        in enumerate(zip(candidates, rerank_scores), start=1)
    ]
    return sorted(ranked, key=lambda result: result[1], reverse=True)[:k_final]

# Print results for each filter using the reranker output format
for case, filters in (("A", {"section": "article"}), ("B", {"source": "class_transcript"})):
    print(f"Case {case}: {filters}")
    results = rerank_filtered("Which AI practices are prohibited?", filters=filters)
    for new_position, (chunk, rerank_score, similarity, search_position) in enumerate(results, start=1):
        source = chunk.metadata.get("source", "-")
        page = chunk.metadata.get("page", "-")
        section = chunk.metadata.get("section", "-")
        print(f"{new_position}. search={search_position} | reranker={rerank_score:.3f} | similarity={similarity:.3f} | source={source} | page={page} | section={section} | {chunk.page_content[:200]}")


recital: 310
article: 403
annex: 62
class: 153
Case A: {'section': 'article'}
1. search=1 | reranker=8.004 | similarity=0.714 | source=law | page=51 | section=article | Article 4
AI literacy
Providers and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of 
their staff and other persons dealing with the op
2. search=19 | reranker=-0.077 | similarity=0.618 | source=law | page=85 | section=article | 2. The obligations set out in paragraph 1, points (a) and (b), shall not apply to providers of AI models that are released 
under a free and open-source licence that allows for the access, usage, modi
3. search=7 | reranker=-0.695 | similarity=0.642 | source=law | page=113 | section=article | industry best practices allowing for the application of such requirements.
2. The AI Office and the Member States shall facilitate the drawing up of codes of conduct concerning the voluntary 
applicat
4. search=8 | reranker=-1.360 | similarity

In [8]:
%pip install -q cohere python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [10]:
# Load the API key without displaying it and create the Cohere client
import os
import cohere
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("COHERE_API_KEY")
print(bool(api_key))
co = cohere.ClientV2(api_key=api_key)

# Retrieve five chunks, build cited context, and ask Cohere for an answer
def answer(query, use_reranker=True, filters=None):
    ranked = (
        rerank_filtered(query, filters=filters, k_final=5)
        if use_reranker else search_filtered(query, k=5, filters=filters)
    )
    chunks_used = [result[0] for result in ranked]
    context = "\n\n".join(
        f"Chunk {index} | source={chunk.metadata.get('source', '-')} | page={chunk.metadata.get('page', '-')} | section={chunk.metadata.get('section', '-')}\n{chunk.page_content}"
        for index, chunk in enumerate(chunks_used, start=1)
    )
    response = co.chat(
        model="command-a-03-2025",
        temperature=0,
        messages=[
            {"role": "system", "content": "Responde solo con el contexto dado. Cita la p\u00e1gina entre corchetes en cada afirmaci\u00f3n. Si la respuesta no est\u00e1 en el contexto, di: No puedo responder con estos documentos."},
            {"role": "user", "content": f"Pregunta: {query}\n\nContexto:\n{context}"},
        ],
    )
    return response.message.content[0].text, chunks_used

# Compare retrieval with and without reranking
query = "Which AI practices are prohibited?"
for label, use_reranker in (("Sin reranker", False), ("Con reranker", True)):
    response_text, used_chunks = answer(query, use_reranker=use_reranker)
    print(label)
    for chunk in used_chunks:
        print(f"source={chunk.metadata.get('source', '-')} | page={chunk.metadata.get('page', '-')} | section={chunk.metadata.get('section', '-')}")
    print(response_text)


True
Sin reranker
source=law | page=51 | section=article
source=law | page=6 | section=recital
source=law | page=7 | section=recital
source=law | page=14 | section=recital
source=law | page=51 | section=article
According to the provided context, the following AI practices are prohibited [page 51]:

1. **The placing on the market, the putting into service, or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behavior of a person or a group of persons by appreciably impairing their ability to make an** [page 51].

Additionally, the context mentions that certain unacceptable AI practices should be prohibited as part of a risk-based approach to regulate AI systems [page 7]. However, the specific details of these prohibited practices are not fully outlined in the provided chunks, except for the one mentioned above.
Con reranker
source=